# P01 · Leer, explorar y verificar PQRS con pandas

Clase 2. Ejecutar una celda a la vez y observar su resultado antes de avanzar. Usamos las tres muestras incluidas, no los completos. No se importan scripts del curso ni se crean funciones de automatización.

**Preparación:** seleccionar el kernel del `.venv` del curso. La ruta de la siguiente celda corresponde a Ubuntu en WSL; si la copia está en otro lugar, cambiar esa única cadena de texto. La ubicación del archivo notebook no cambia esa ruta. Los CSV originales permanecen intactos.

In [ ]:
import pandas as pd

repositorio = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
archivo_2023_II = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2023_II.csv"
archivo_2024_I = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2024_I.csv"
archivo_2024_II = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2024_II.csv"
print(archivo_2023_II)
print(archivo_2024_I)
print(archivo_2024_II)
print("pandas:", pd.__version__)

## 1. Mirar el archivo antes de leerlo
`open` permite inspeccionar la cabecera. El manifiesto describe la procedencia, los separadores y el esquema esperado. `json.load` solo lee ese documento, no transforma los CSV.

In [ ]:
with open(archivo_2023_II, encoding="utf-8-sig") as archivo:
    print(archivo.readline())
with open(archivo_2024_I, encoding="utf-8-sig") as archivo:
    print(archivo.readline())
with open(archivo_2024_II, encoding="utf-8-sig") as archivo:
    print(archivo.readline())

In [ ]:
import json
with open(repositorio + "/kit/pqrs_fuentes.json", encoding="utf-8") as archivo:
    manifiesto = json.load(archivo)
df_manifiesto = pd.DataFrame(manifiesto).set_index("periodo")
display(df_manifiesto[["archivo", "separador", "sha256_muestra"]])

**Antes de continuar:** identifica qué archivo usa coma y cuáles usan punto y coma. Predice qué pasaría si usaras el separador incorrecto.

## 2. Leer cada CSV explícitamente
`sep` indica el separador; `dtype=str` conserva códigos y ceros iniciales; `keep_default_na=False` conserva textos vacíos y marcadores en lugar de interpretarlos automáticamente como nulos. `encoding="utf-8-sig"` admite la marca BOM.

In [ ]:
df_2023_II = pd.read_csv(
    archivo_2023_II,
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(df_2023_II.head())
print(df_2023_II.shape)

In [ ]:
df_2024_I = pd.read_csv(
    archivo_2024_I,
    sep=",",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(df_2024_I.head())
print(df_2024_I.shape)

In [ ]:
df_2024_II = pd.read_csv(
    archivo_2024_II,
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(df_2024_II.head())
print(df_2024_II.shape)

## 3. Inspeccionar y comparar los esquemas
Cada muestra debe tener 1.000 filas y 38 columnas. Primero vemos los resultados; después escribimos `assert` para detenernos si no se cumple el contrato. No basta con contar columnas: también comparamos nombres y orden.

In [ ]:
df_2023_II.info()
display(df_2023_II.dtypes)
print(df_2023_II.columns.tolist())
print("2023 frente a 2024-I:", df_2023_II.columns.equals(df_2024_I.columns))
print("2023 frente a 2024-II:", df_2023_II.columns.equals(df_2024_II.columns))
assert df_2023_II.shape == (1000, 38)
assert df_2024_I.shape == (1000, 38)
assert df_2024_II.shape == (1000, 38)
assert df_2023_II.columns.tolist() == df_manifiesto.loc["2023_II", "columnas"]
assert df_2024_I.columns.tolist() == df_manifiesto.loc["2024_I", "columnas"]
assert df_2024_II.columns.tolist() == df_manifiesto.loc["2024_II", "columnas"]

## 4. Seleccionar campos y reconocer faltantes
Prueba primero sobre un campo; luego cambia el nombre por otro que quieras investigar. Vacío, nulo y `ND` se registran por separado. No reemplazamos ninguno por cero.

In [ ]:
campo = "pet_cod_mpio"
valores = df_2023_II[campo].str.strip()
print("Nulos:", valores.isna().sum())
print("Textos vacíos:", valores.eq("").sum())
print("Marcadores ND:", valores.str.upper().eq("ND").sum())
display(valores.value_counts(dropna=False).head(10))
display(df_2023_II.loc[valores.eq(""), ["OBJECTID", "pet_cod_mpio", "afec_cod_mpio"]])

## 5. Unir muestras conservando su procedencia
`assign` crea una copia con la columna del corte; `concat` apila filas. `periodo` es el año declarado en la fila, mientras que `corte_archivo` identifica de qué publicación vino. No confundir ambos.

In [ ]:
df_pqrs = pd.concat([
    df_2023_II.assign(corte_archivo="2023_II"),
    df_2024_I.assign(corte_archivo="2024_I"),
    df_2024_II.assign(corte_archivo="2024_II")
], ignore_index=True)
print(df_pqrs.shape)
assert len(df_pqrs) == 3000
display(df_pqrs.groupby(["corte_archivo", "periodo", "mes"]).size().rename("filas").reset_index())

In [ ]:
campos = ["periodo", "mes", "pet_cod_mpio", "afec_cod_mpio", "OBJECTID"]
perfil = pd.DataFrame(index=campos)
perfil["nulos"] = df_pqrs[campos].isna().sum()
# El bucle repite operaciones que ya observamos sobre una sola columna.
for campo in campos:
    valores = df_pqrs[campo].str.strip()
    perfil.loc[campo, "vacios"] = valores.eq("").sum()
    perfil.loc[campo, "ND"] = valores.str.upper().eq("ND").sum()
display(perfil)
es_repetido = df_pqrs.duplicated(["corte_archivo", "OBJECTID"], keep=False)
print("Filas con clave candidata repetida:", es_repetido.sum())
display(df_pqrs.loc[es_repetido, ["corte_archivo", "OBJECTID"]])

**Interpretación:** peticionario y afectado pueden tener ubicaciones distintas. `id_afec` no es una clave de reclamo; corte–OBJECTID es una clave candidata, no una prueba de identidad semántica. Describe lo que observas antes de proponer una corrección.

## 6. Detectar valores no convertibles sin perderlos
La lectura como texto permite inspeccionar los valores. Creamos una serie numérica separada y distinguimos un vacío de un texto no convertible.

In [ ]:
mes_texto = df_pqrs["mes"].str.strip()
mes_numero = pd.to_numeric(mes_texto, errors="coerce")
es_vacio = mes_texto.eq("") | mes_texto.isna()
no_convertible = mes_numero.isna() & ~es_vacio
fuera_rango = mes_numero.notna() & (~mes_numero.between(1, 12) | mes_numero.mod(1).ne(0))
df_revision_mes = df_pqrs.loc[es_vacio | no_convertible | fuera_rango, ["corte_archivo", "OBJECTID", "mes"]].copy()
print("Vacíos:", es_vacio.sum(), "No convertibles:", no_convertible.sum(), "Fuera de dominio:", fuera_rango.sum())
display(df_revision_mes)
# df_pqrs conserva mes como texto; no se descartaron filas.

## 7. Leer por bloques: observar antes de repetir
`chunksize` devuelve un lector de bloques. `next` toma solamente el primero. Modifica `tamano_bloque` y vuelve a ejecutar esta celda para ver qué cambia. Los bytes del DataFrame no son la memoria total del proceso (RSS).

In [ ]:
tamano_bloque = 100
with pd.read_csv(archivo_2023_II, sep=";", encoding="utf-8-sig", dtype=str,
                 keep_default_na=False, chunksize=tamano_bloque) as lector:
    primer_bloque = next(lector)
display(primer_bloque.head())
print("Filas del bloque:", len(primer_bloque))
print("Bytes del DataFrame:", primer_bloque.memory_usage(deep=True).sum())

Ahora repetimos con 100, 500 y 1.000 filas sobre **los tres archivos**. El alumno puede modificar tamaños y observar cada bloque. El bucle es visible: suma filas, acumula vacíos y guarda la memoria máxima observada. Cada combinación se compara con la lectura completa, no solo con otra ejecución por bloques.

In [ ]:
mediciones = []
archivos = [archivo_2023_II, archivo_2024_I, archivo_2024_II]
separadores = [";", ",", ";"]
dataframes = [df_2023_II, df_2024_I, df_2024_II]
cortes = ["2023_II", "2024_I", "2024_II"]

for posicion in range(3):
    for tamano_bloque in [100, 500, 1000]:
        total_filas = 0
        total_vacios = pd.Series(0, index=dataframes[posicion].columns)
        memoria_maxima = 0
        with pd.read_csv(archivos[posicion], sep=separadores[posicion], encoding="utf-8-sig",
                         dtype=str, keep_default_na=False, chunksize=tamano_bloque) as lector:
            for bloque in lector:
                total_filas += len(bloque)
                memoria_bloque = int(bloque.memory_usage(deep=True).sum())
                memoria_maxima = max(memoria_maxima, memoria_bloque)
                for campo in bloque.columns:
                    total_vacios[campo] += bloque[campo].str.strip().eq("").sum()
        assert total_filas == len(dataframes[posicion])
        for campo in total_vacios.index:
            assert total_vacios[campo] == dataframes[posicion][campo].str.strip().eq("").sum()
        mediciones.append({"corte": cortes[posicion], "bloque": tamano_bloque,
                           "filas": total_filas, "memoria_dataframe_max_bytes": memoria_maxima})

comparacion_bloques = pd.DataFrame(mediciones)
display(comparacion_bloques)

## 8. Verificar el archivo: bytes y SHA-256
Pandas verifica datos; SHA-256 verifica bytes del archivo. Para estas muestras pequeñas leemos cada archivo en memoria con `read()`. Para los completos, más adelante se implementará lectura binaria por bloques. El hash de la muestra se compara con `sha256_muestra`, no con el de su archivo completo.

In [ ]:
import hashlib

huellas = []
for posicion in range(3):
    with open(archivos[posicion], "rb") as archivo:
        contenido = archivo.read()
    calculado = hashlib.sha256(contenido).hexdigest()
    esperado = df_manifiesto.loc[cortes[posicion], "sha256_muestra"]
    huellas.append({"corte": cortes[posicion], "bytes_archivo": len(contenido),
                    "sha256_calculado": calculado, "coincide": calculado == esperado})
    assert calculado == esperado, "Revisar el origen del archivo antes de continuar"
df_huellas = pd.DataFrame(huellas)
display(df_huellas)

## 9. Guardar una evidencia elegida por el alumno
Comprueba la ruta impresa antes de escribir. Estas instrucciones guardan únicamente tablas de control; repetirlas reemplaza esos CSV. Cambia el nombre si quieres conservar otra ejecución.

In [ ]:
import os
carpeta_salida = repositorio + "/kit/salidas/clase2/P01"
print("Guardaré en:", carpeta_salida)
os.makedirs(carpeta_salida, exist_ok=True)
perfil.to_csv(carpeta_salida + "/perfil_campos.csv", index_label="campo")
comparacion_bloques.to_csv(carpeta_salida + "/comparacion_bloques.csv", index=False)
df_huellas.to_csv(carpeta_salida + "/verificacion_archivos.csv", index=False)
df_revision_mes.to_csv(carpeta_salida + "/revision_mes.csv", index=False)

## Entrega e interpretación propia

1. Explica separadores, esquema y tipos de los tres archivos.
2. Escribe un diccionario de cinco campos y distingue peticionario, afectado y reporte.
3. Compara memoria y conteos por tamaño de bloque; no extrapoles RSS a partir del CSV.
4. Justifica cómo tratarías faltantes y valores no convertibles sin borrar evidencia.
5. Declara dos límites: primeras 1.000 filas no aleatorias y reportes que no equivalen a prevalencia clínica.

Para archivos completos se verificarían también hashes, esquema, filas y anchos de registro con lectura por bloques. Las funciones que automatizan estos pasos se estudiarán después de la clase 2.